<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-9_Trajectory_%ED%8F%89%EA%B0%80_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 9 — Trajectory 평가 (Agent 실행 경로 분석)

**학습 목표**  
1. Agent의 최종 답변뿐 아니라 **실행 경로(trajectory)**를 정량적으로 평가합니다.  
2. `steps`, `tool_calls`, `repeated_tool_types`를 로그에서 직접 계산합니다.  
3. 성공률, 토큰, 비용, 지연시간, 재시도, 허용되지 않은 행동, 인용 품질의 차이를 이해합니다.

✅ **Google Colab에서 바로 실행 가능** · 별도 설치/API 키 불필요 · 교육용 예제 데이터 사용

> **주의:** 단순히 같은 도구를 다시 호출했다고 오류인 것은 아닙니다. `repeated_tool_types`는 비효율성의 단서일 뿐, 품질 판정은 실행 맥락과 함께 해야 합니다.

## 1. 예제 실행 로그 만들기

업로드한 자료의 `row["action"]["tool"]` 구조를 그대로 활용합니다. 각 단계에는 시간과 모델 사용량 같은 **가상 측정값**을 추가했습니다. 여기서 `tokens`는 실제 API 사용량이 아닙니다.

In [1]:
# 질문: "지능시스템 과제 비율이 몇 %인가요?"
# 실제 에이전트를 호출하는 대신 수업용 실행 로그를 직접 정의합니다.

successful_log = [
    {"action": {"tool": "search_notes", "args": {"query": "지능시스템 평가"}},
     "result": "과제 20%, 중간 30%, 기말 30%, 출석 20%", "tokens": 180, "latency_ms": 310},
    {"action": {"tool": "open_document", "args": {"doc_id": "course_eval"}},
     "result": "과제 20% (강의 평가 안내)", "tokens": 120, "latency_ms": 210},
    {"action": {"tool": "finish", "answer": "과제 반영 비율은 20%입니다."},
     "result": None, "tokens": 90, "latency_ms": 140},
]

repetitive_log = [
    {"action": {"tool": "search_notes", "args": {"query": "평가"}},
     "result": "관련 문서 1건", "tokens": 180, "latency_ms": 300},
    {"action": {"tool": "search_notes", "args": {"query": "과제"}},
     "result": "관련 문서 1건", "tokens": 180, "latency_ms": 300},
    {"action": {"tool": "search_notes", "args": {"query": "과제 비율"}},
     "result": "과제 20%", "tokens": 160, "latency_ms": 270},
    {"action": {"tool": "finish", "answer": "과제 반영 비율은 20%입니다."},
     "result": None, "tokens": 100, "latency_ms": 160},
]
print("정상 예제 단계 수:", len(successful_log))
print("반복 검색 예제 단계 수:", len(repetitive_log))

정상 예제 단계 수: 3
반복 검색 예제 단계 수: 4


## 2. `trajectory_metrics()` 구현

- `steps`: 전체 로그 행 수 (여기서는 **finish 포함**)
- `tool_calls`: `finish`를 제외한 도구 실행 수
- `repeated_tool_types`: `도구 호출 총수 − 고유 도구 종류 수` (동일 이름 재사용 횟수)

`repeated_tool_types`는 **정확히 같은 인자로 재호출한 횟수와는 다릅니다**.

In [2]:
def trajectory_metrics(log):
    """Agent의 실행 로그에서 기본 trajectory 지표를 계산합니다."""
    tool_steps = [
        row
        for row in log
        if row.get("action", {}).get("tool") not in (None, "finish")
    ]

    # 도구 이름만 모읍니다. 예: ["search_notes", "search_notes", "open_document"]
    tools = [row["action"]["tool"] for row in tool_steps]

    # 반복된 도구 이름의 개수: 호출 3회, 종류 2개라면 1
    repeated = len(tools) - len(set(tools))
    return {
        "steps": len(log),
        "tool_calls": len(tool_steps),
        "repeated_tool_types": repeated,
    }

print("효율적인 경로:", trajectory_metrics(successful_log))
print("반복 검색 경로:", trajectory_metrics(repetitive_log))

효율적인 경로: {'steps': 3, 'tool_calls': 2, 'repeated_tool_types': 0}
반복 검색 경로: {'steps': 4, 'tool_calls': 3, 'repeated_tool_types': 2}


## 3. 반복 도구 유형과 동일 호출 비교

같은 도구를 사용해도 검색어가 달라졌다면 전략적으로 합리적인 추가 검색일 수 있습니다. **같은 도구 + 같은 인자**를 사용한 반복 호출을 별도로 세어봅니다.

In [3]:
import json

def duplicate_exact_calls(log):
    """도구 이름과 인자가 모두 동일한 재호출의 횟수를 계산합니다."""
    signatures = []
    for row in log:
        action = row.get("action", {})
        tool = action.get("tool")
        if tool in (None, "finish"):
            continue
        # JSON 문자열로 바꾸면 사전(dict)의 키 순서에 영향받지 않습니다.
        args_text = json.dumps(action.get("args", {}), ensure_ascii=False, sort_keys=True)
        signatures.append((tool, args_text))
    return len(signatures) - len(set(signatures))

print("반복 검색 예제의 도구 종류 반복:", trajectory_metrics(repetitive_log)["repeated_tool_types"])
print("반복 검색 예제의 완전히 동일한 호출 반복:", duplicate_exact_calls(repetitive_log))

exact_repeat_log = repetitive_log[:2] + [repetitive_log[0]] + repetitive_log[2:]
print("동일 호출을 한 번 더 넣은 예제:", duplicate_exact_calls(exact_repeat_log))

반복 검색 예제의 도구 종류 반복: 2
반복 검색 예제의 완전히 동일한 호출 반복: 0
동일 호출을 한 번 더 넣은 예제: 1


## 4. 평가할 항목 확장하기

이미지에 제시된 실제 평가 항목은 **Success, Total Tokens, Cost, Latency, Unauthorized Action, Invalid Tool Call, Retry Count, Citation Quality**입니다.

이번 노트북은 다음과 같이 취급합니다.

- **Success**: 정답 여부를 미리 준비한 정답과 비교한 **교육용 자동 평가** (현업의 정답 판정은 별도 평가 설계 필요)
- **Total Tokens / Latency**: 로그에 기록된 측정값만 더합니다. 예제 값은 가상 데이터입니다.
- **Cost**: 예제에서 정한 가상 단가를 사용한 추정값일 뿐 실제 Gemini 청구 비용이 아닙니다.
- **Unauthorized Action / Invalid Tool Call / Retry Count**: 별도 이벤트 로그가 있어야 측정할 수 있습니다.
- **Citation Quality**: 출처의 실제 적합성과 근거 일치를 검사해야 하므로 이 실습에서는 사람이 검토하는 항목으로 남깁니다.

In [4]:
ALLOWED_TOOLS = {"search_notes", "open_document", "calculate"}
# 교육용 가정: 1,000 tokens 당 $0.001. 실제 모델 API 요금이 아닙니다.
DEMO_USD_PER_1000_TOKENS = 0.001

def evaluate_run(log, expected_answer, *, denied_events=None, retry_count=0):
    """기본 trajectory 지표와 교육용 추가 평가값을 합칩니다."""
    denied_events = denied_events or []
    base = trajectory_metrics(log)

    # 마지막 finish의 답변 추출. finish가 없을 수도 있습니다.
    finish_actions = [row["action"] for row in log
                      if row.get("action", {}).get("tool") == "finish"]
    answer = finish_actions[-1].get("answer") if finish_actions else None

    total_tokens = sum(row.get("tokens", 0) for row in log)
    latency_ms = sum(row.get("latency_ms", 0) for row in log)

    # 차단된 시도와 실제 실행 기록은 별도입니다. 두 값을 혼동하지 않습니다.
    invalid_tool_calls = sum(
        event.get("reason") == "invalid_tool" for event in denied_events
    )
    unauthorized_actions = sum(
        event.get("reason") == "unauthorized_action" for event in denied_events
    )

    return {
        **base,
        "success": answer == expected_answer,
        "total_tokens": total_tokens,
        "estimated_cost_usd_demo": round(total_tokens / 1000 * DEMO_USD_PER_1000_TOKENS, 6),
        "latency_ms": latency_ms,
        "unauthorized_actions": unauthorized_actions,
        "invalid_tool_calls": invalid_tool_calls,
        "retry_count": retry_count,
        "duplicate_exact_calls": duplicate_exact_calls(log),
        "citation_quality": "not_evaluated",  # 사람/별도 평가기 확인 필요
    }

EXPECTED = "과제 반영 비율은 20%입니다."
fast_metrics = evaluate_run(successful_log, EXPECTED)
slow_metrics = evaluate_run(repetitive_log, EXPECTED)
for name, result in [("효율적인 경로", fast_metrics), ("반복 검색 경로", slow_metrics)]:
    print("\n", name)
    for key, value in result.items():
        print(f"  {key}: {value}")


 효율적인 경로
  steps: 3
  tool_calls: 2
  repeated_tool_types: 0
  success: True
  total_tokens: 390
  estimated_cost_usd_demo: 0.00039
  latency_ms: 660
  unauthorized_actions: 0
  invalid_tool_calls: 0
  retry_count: 0
  duplicate_exact_calls: 0
  citation_quality: not_evaluated

 반복 검색 경로
  steps: 4
  tool_calls: 3
  repeated_tool_types: 2
  success: True
  total_tokens: 620
  estimated_cost_usd_demo: 0.00062
  latency_ms: 1030
  unauthorized_actions: 0
  invalid_tool_calls: 0
  retry_count: 0
  duplicate_exact_calls: 0
  citation_quality: not_evaluated


## 5. 성공과 효율성 비교

두 실행 경로가 같은 최종 답변을 냈더라도 도구 호출 수, 토큰, 지연시간은 다를 수 있습니다. `success`가 같다고 trajectory 품질이 같은 것은 아닙니다.

In [5]:
# 표 형태로 간단하게 출력합니다. (별도 패키지 설치 필요 없음)
columns = ["success", "steps", "tool_calls", "repeated_tool_types", "total_tokens", "latency_ms"]
print("metric".ljust(23), "효율적인 경로".ljust(17), "반복 검색 경로")
print("-" * 65)
for key in columns:
    print(key.ljust(23), str(fast_metrics[key]).ljust(17), slow_metrics[key])

metric                  효율적인 경로           반복 검색 경로
-----------------------------------------------------------------
success                 True              True
steps                   3                 4
tool_calls              2                 3
repeated_tool_types     0                 2
total_tokens            390               620
latency_ms              660               1030


## 6. 안전성 및 오류 이벤트는 별도로 기록

**차단된** 불법적인 도구 호출은 실제 실행 로그에 남지 않을 수도 있으므로, Harness는 별도의 이벤트 로그를 기록해야 합니다. 단, 차단 이벤트 수는 실제 실행 횟수와 다릅니다.

In [6]:
# 이 데이터는 교육용으로 만들어 놓은 차단 시도 예제입니다.
denied_events = [
    {"tool": "delete_all_files", "reason": "invalid_tool"},
    {"tool": "open_document", "reason": "unauthorized_action"},
]

audit_metrics = evaluate_run(
    successful_log,
    EXPECTED,
    denied_events=denied_events,
    retry_count=1,
)
print("차단된 유효하지 않은 도구 호출:", audit_metrics["invalid_tool_calls"])
print("권한 위반 시도:", audit_metrics["unauthorized_actions"])
print("재시도 횟수:", audit_metrics["retry_count"])
print("인용 품질:", audit_metrics["citation_quality"])

차단된 유효하지 않은 도구 호출: 1
권한 위반 시도: 1
재시도 횟수: 1
인용 품질: not_evaluated
